# Orchestration Projects

**Module:** 14 — AI Orchestration

Build a support ticket orchestrator and a research crew workflow.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Implement two end-to-end orchestration mini-projects
- Include HITL, state, and tests in your deliverables
- Document SLOs and failure modes


## Project 1 — Support Ticket Orchestrator

**Brief:** Ingest a ticket → redact → parallel enrich → draft → HITL if needed → send.

```mermaid
flowchart TD
  A[Ingest] --> B[Redact]
  B --> C[KB]
  B --> D[CRM]
  B --> E[Sentiment]
  C --> F[Draft]
  D --> F
  E --> F
  F --> G{risk/high $?}
  G -->|yes| H[HITL]
  G -->|no| I[Send]
  H --> I
```

| ID | Acceptance |
|----|------------|
| T1 | PII email redacted before model call |
| T2 | Parallel enrich completes with merge |
| T3 | Refund > $100 pauses for approval |
| T4 | Checkpoint resume after approval |


In [ ]:
# Project 1 starter
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy

def redact(text): return text.replace("@", "[at]")

def enrich(text):
    with ThreadPoolExecutor(max_workers=3) as ex:
        kb = ex.submit(lambda: ["kb:refund_policy"])
        crm = ex.submit(lambda: {"tier": "pro"})
        sent = ex.submit(lambda: "neg" if "angry" in text.lower() else "neu")
        return {"kb": kb.result(), "crm": crm.result(), "sentiment": sent.result()}

def draft(text, enrich_):
    return f"Hello, thanks for contacting us about '{text[:40]}' ({enrich_['sentiment']})."

def orchestrate(ticket, approved=False, amount=0):
    state = {"text": redact(ticket), "cursor": "enrich"}
    state["enrich"] = enrich(state["text"])
    state["draft"] = draft(state["text"], state["enrich"])
    if amount >= 100 and not approved:
        state["status"] = "waiting_human"
        return state
    state["status"] = "sent"
    return state

print(orchestrate("Angry: charge me wrong email a@b.com", amount=150))
print(orchestrate("Angry: charge me wrong email a@b.com", amount=150, approved=True)["status"])


### Try it yourself — Project 1

1. Persist checkpoints in a dict store keyed by ticket_id.
2. Add idempotent `send` so retries don't double-email.
3. Write asserts for T1–T4.


## Project 2 — Research Crew

**Brief:** Manager plans → researcher retrieves → writer drafts → critic scores → optional loop.

| Role | Output |
|------|--------|
| Manager | Outline JSON |
| Researcher | Sources list |
| Writer | Markdown report |
| Critic | Score + gaps |

**Constraints:** max 2 critique loops; budget $0.20; cite ≥ 3 sources.


In [ ]:
# Project 2 starter
def manager(goal):
    return {"sections": ["background", "findings", "risks"], "goal": goal}

def researcher(outline):
    return [f"source:{s}" for s in outline["sections"]]

def writer(outline, sources):
    return "# Report\n" + "\n".join(f"## {s}\n(based on source:{s})" for s in outline["sections"])

def critic(report, sources):
    score = 0.7 if len(sources) >= 3 else 0.4
    gaps = [] if score >= 0.7 else ["need more sources"]
    return {"score": score, "gaps": gaps}

def research_crew(goal, max_loops=2):
    outline = manager(goal)
    sources = researcher(outline)
    report = writer(outline, sources)
    for i in range(max_loops):
        c = critic(report, sources)
        if c["score"] >= 0.7:
            return {"report": report, "crit": c, "loops": i}
        sources.append(f"source:extra{i}")
        report = writer(outline, sources)
    return {"report": report, "crit": critic(report, sources), "loops": max_loops}

print(research_crew("AI orchestration risks")["crit"])


## Deliverables Checklist
- [ ] Architecture diagram
- [ ] State schema + version field
- [ ] HITL or critic loop demonstrated
- [ ] Budget guard
- [ ] ≥5 automated tests
- [ ] SLO sheet (targets + how measured)
- [ ] Failure mode list with mitigations


In [ ]:
# Shared test examples
assert "@" not in orchestrate("mail me at x@y.com")["text"]
out = research_crew("test")
assert out["crit"]["score"] >= 0.7
print("smoke tests passed")


### Try it yourself — Ship

1. Add tracing spans to Project 1.
2. Add a cost accumulator to Project 2 and abort if > budget.

**Stretch:** Swap critic for a human review event in Project 2.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `crew` | Role-specialized multi-agent team |
| `critic loop` | Generate→evaluate→revise cycle |
| `acceptance` | Binary project requirement check |


## Acceptance Test Matrix

| Test | Project | Assert |
|------|---------|--------|
| PII | 1 | `@` not in model-facing text |
| Parallel merge | 1 | kb+crm+sentiment present |
| HITL threshold | 1 | amount>=100 ⇒ waiting_human |
| Resume | 1 | after approve ⇒ sent |
| Critic loop bound | 2 | loops ≤ max_loops |
| Citations | 2 | ≥3 sources |
| Budget | 2 | abort when exceeded |

### Grading demo script
Run tests → print architecture ASCII → show one intentional failure + mitigation.


In [ ]:
# Budgeted research crew
class Budget:
    def __init__(self, max_usd): self.max=max_usd; self.spent=0
    def use(self, u):
        if self.spent + u > self.max: raise RuntimeError("budget")
        self.spent += u

def crew(goal, budget=Budget(0.05)):
    budget.use(0.01)  # plan
    sources = []
    for i in range(5):
        budget.use(0.01)  # retrieve
        sources.append(f"s{i}")
        if len(sources) >= 3:
            break
    budget.use(0.02)  # write
    return {"sources": sources, "spent": budget.spent}

print(crew("x"))
try:
    crew("y", Budget(0.02))
except RuntimeError as e:
    print("caught", e)


### Try it yourself — Ship deepen

1. Add OpenTelemetry-like spans around Project 1 steps.
2. Write a chaos test: tool `crm` raises once, ensure retry then success.


## Portfolio README cells (inside notebook only)
Document: architecture, setup env vars (`YOUR_OPENAI_API_KEY`), how to run tests, SLO table, known limitations.


## Key Takeaways

- Projects should exercise patterns+state+HITL
- Tests beat screenshots
- Document SLOs even for homework systems
